# Exposing the compressor nonlinearity (Output-Transformer dataset split)

A companion to `eval_output_transformer_h1_coherence.ipynb`. There the
magnitude-squared coherence sat at ~0.99 for every setting — **not** because the
compressor is linear, but because ordinary coherence is the wrong lens for it.

A compressor is, to first order, a **slowly time-varying broadband real gain**
`g(t)`. Within one short STFT window the wet is `Y(f) ≈ g·X(f)` — a scalar,
frequency-flat multiple of the input — so at the per-bin / matched-time level the
dry→wet relation is almost perfectly *linear*, and `γ² → 1`. The nonlinearity
lives in **how `g(t)` is driven by the signal envelope** (threshold / ratio /
attack / release) and in the harmonic coloration it adds, neither of which a
two-signal linear coherence can see.

This notebook reuses the **exact same dataset split and streaming machinery** and
applies the four measures that *do* surface the nonlinearity:

1. **Best-linear-model residual / NMSE** — fraction of output power that *no*
   per-frequency LTI filter can explain `= Σ Syy(1−γ²) / Σ Syy`. (True test-tone
   THD is not measurable here — this is recorded program material, no controlled
   sines run through the box — so this model-based residual is the data-driven
   analogue.)
2. **Gain-matched residual spectrum** — apply the known broadband GR envelope
   `g(t)`, then look at the spectrum of `r = wet − g·dry`. With the level/dynamics
   removed, what remains is the residual **coloration + harmonic distortion**.
3. **Envelope multiple-coherence** — add the envelope-modulated input `m = g·dry`
   as a *second* regressor and compute the multiple coherence `γ²_{y:x,m}`. The
   **gap** over ordinary coherence is the output power explained only by the
   time-varying gain — a direct signature of the envelope-driven nonlinearity.
4. **Bicoherence** — a higher-order (third-moment) spectrum that detects
   **quadratic phase coupling**. Output bicoherence elevated above input
   bicoherence is unambiguous nonlinearly-generated structure.

All STFT framing uses **librosa**, matching the coherence notebook.


In [ ]:
import os
import sys
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import librosa
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)

REPO_ROOT = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file()),
    Path.cwd().resolve(),
)

sys.path.insert(0, str(REPO_ROOT / "06_output"))
sys.path.insert(0, str(REPO_ROOT / "03_initial_GR_pred"))

try:
    from dataset import discover_output_transformer_pairs
except ModuleNotFoundError as e:
    if "lightning" not in str(e).lower():
        raise
    import glob

    def discover_output_transformer_pairs(data_root: str) -> list[dict]:
        """Fallback discovery (no Lightning import)."""
        dry_lookup = {
            os.path.basename(p).replace("_UnmasteredWAV.wav", ""): p
            for p in glob.glob(
                os.path.join(data_root, "processed_normalized", "*_UnmasteredWAV.wav")
            )
        }
        pairs: list[dict] = []
        gr_root = os.path.join(data_root, "gr_curves")
        for setting in sorted(os.listdir(gr_root)):
            if not setting.startswith("threshold_"):
                continue
            wet_dir = os.path.join(data_root, "processed_ground_truth", setting)
            for pt in sorted(glob.glob(os.path.join(gr_root, setting, "*.pt"))):
                song = os.path.splitext(os.path.basename(pt))[0]
                wet = os.path.join(wet_dir, f"{song}-exported.wav")
                if song in dry_lookup and os.path.isfile(wet):
                    pairs.append(
                        {"song": song, "setting": setting, "dry": dry_lookup[song], "gr": pt, "wet": wet}
                    )
        return sorted(pairs, key=lambda p: (p["song"], p["setting"]))

    print("[warn] Using fallback discover_output_transformer_pairs(): missing Lightning")

from splits import build_split_manifest
from amplitude_match import GR_DB_MIN, GR_DB_MAX, gr_db_to_gain
from eval_helpers import _read_dry_wet_segment, _pair_num_frames

print(f"repo    : {REPO_ROOT}")
print(f"torch   : {torch.__version__}")
print(f"librosa : {librosa.__version__}")

In [ ]:
# Diff-SSL-G-Comp  (identical to the coherence notebook)
DATA_ROOT = "/Volumes/Saola\'s Drive/AllCode/thesis/data/Diff-SSL-G-Comp"
SAMPLE_RATE = 44100

STREAM_CHUNK_SEC = 10.0

# STFT / spectral-averaging parameters (librosa)
N_FFT = 4096
HOP_LENGTH = N_FFT // 4
WINDOW = "hann"
EPS = 1e-12

MAX_EVAL_PAIRS = None

assert os.path.isdir(DATA_ROOT), f"Missing DATA_ROOT: {DATA_ROOT}"
assert os.path.isdir(os.path.join(DATA_ROOT, "gr_curves")), "Missing gr_curves folder"

pairs = discover_output_transformer_pairs(DATA_ROOT)
print(f"Discovered (song, setting) pairs: {len(pairs)}")

SPLIT_SEED = 42
N_VAL_SONGS = 1
N_TEST_SONGS = 2
split = build_split_manifest(
    pairs, seed=SPLIT_SEED, n_val_songs=N_VAL_SONGS, n_test_songs=N_TEST_SONGS
)

VAL_PAIRS = [tuple(k.split("::", 1)) for k in split.val_pair_keys]
TEST_PAIRS = [tuple(k.split("::", 1)) for k in split.test_pair_keys]

USED_PAIRS = [("validation", s, c) for (s, c) in VAL_PAIRS] + [
    ("test", s, c) for (s, c) in TEST_PAIRS
]
if MAX_EVAL_PAIRS is not None:
    USED_PAIRS = USED_PAIRS[:MAX_EVAL_PAIRS]

SETTINGS = sorted({c for (_, _, c) in USED_PAIRS})
print(f"Val pairs  : {len(VAL_PAIRS)} (songs={split.val_songs})")
print(f"Test pairs : {len(TEST_PAIRS)} (songs={split.test_songs})")
print(f"Used pairs : {len(USED_PAIRS)}")
print(f"Settings   : {len(SETTINGS)}")

chunk_frames = int(round(STREAM_CHUNK_SEC * SAMPLE_RATE))
FREQS = librosa.fft_frequencies(sr=SAMPLE_RATE, n_fft=N_FFT)
print(f"n_fft={N_FFT}, hop={HOP_LENGTH} -> {len(FREQS)} freq bins")

## Shared one-pass accumulator

A single streaming pass per setting computes the STFTs of the **dry** `X`, the
**wet** `Y`, and the **gain-matched** `M = g·dry`, and accumulates every Welch
sum the four measures need:

* `Sxx, Syy, Smm` — auto-spectra,
* `Sxy, Sxm, Smy` — cross-spectra,
* `Srr = Σ|Y − M|²` — the gain-matched residual auto-spectrum (STFT is linear, so
  `STFT(wet − g·dry) = Y − M`).

From these: §1 coherence/NMSE (`Sxx,Syy,Sxy`), §2 residual (`Srr,Syy`), §3
multiple coherence (the full 2-input cross-spectral matrix).

In [ ]:
def pair_paths(song: str, setting: str) -> tuple[str, str, str]:
    dry_path = os.path.join(DATA_ROOT, "processed_normalized", f"{song}_UnmasteredWAV.wav")
    wet_path = os.path.join(DATA_ROOT, "processed_ground_truth", setting, f"{song}-exported.wav")
    gr_path = os.path.join(DATA_ROOT, "gr_curves", setting, f"{song}.pt")
    return dry_path, wet_path, gr_path


def _stft(sig: np.ndarray) -> np.ndarray:
    return librosa.stft(
        np.ascontiguousarray(sig, dtype=np.float32),
        n_fft=N_FFT, hop_length=HOP_LENGTH, window=WINDOW, center=False,
    )


class NLAccumulator:
    """Welch sums for the dry (X), wet (Y) and gain-matched (M) signals."""

    def __init__(self, n_bins: int):
        self.Sxx = np.zeros(n_bins, np.float64)
        self.Syy = np.zeros(n_bins, np.float64)
        self.Smm = np.zeros(n_bins, np.float64)
        self.Sxy = np.zeros(n_bins, np.complex128)
        self.Sxm = np.zeros(n_bins, np.complex128)
        self.Smy = np.zeros(n_bins, np.complex128)
        self.Srr = np.zeros(n_bins, np.float64)   # |Y - M|^2
        self.n_frames = 0

    def add(self, x, y, m):
        if min(len(x), len(y), len(m)) < N_FFT:
            return
        X, Y, M = _stft(x), _stft(y), _stft(m)
        self.Sxx += np.sum(np.abs(X) ** 2, axis=1)
        self.Syy += np.sum(np.abs(Y) ** 2, axis=1)
        self.Smm += np.sum(np.abs(M) ** 2, axis=1)
        self.Sxy += np.sum(np.conj(X) * Y, axis=1)
        self.Sxm += np.sum(np.conj(X) * M, axis=1)
        self.Smy += np.sum(np.conj(M) * Y, axis=1)
        self.Srr += np.sum(np.abs(Y - M) ** 2, axis=1)
        self.n_frames += X.shape[1]

    # §1 -----------------------------------------------------------------
    def coherence(self):
        return (np.abs(self.Sxy) ** 2) / (self.Sxx * self.Syy + EPS)

    def linear_unexplained(self):
        """Per-bin output power fraction no LTI filter can explain = 1 - gamma^2."""
        return 1.0 - self.coherence()

    # §2 -----------------------------------------------------------------
    def distortion_ratio(self):
        """Gain-matched residual / output power, per bin (linear ratio)."""
        return self.Srr / (self.Syy + EPS)

    # §3 -----------------------------------------------------------------
    def multiple_coherence(self, ridge=1e-9):
        """gamma^2_{y:(x,m)} via per-bin 2x2 cross-spectral solve."""
        a = self.Sxx + ridge
        d = self.Smm + ridge
        b = self.Sxm                     # c = conj(b)
        det = a * d - np.abs(b) ** 2 + EPS
        p, q = self.Sxy, self.Smy        # b_vec = [Sxy, Smy]
        inv0 = (d * p - b * q) / det                 # (Sxx^-1 b_vec)[0]
        inv1 = (-np.conj(b) * p + a * q) / det       # (Sxx^-1 b_vec)[1]
        num = np.real(np.conj(p) * inv0 + np.conj(q) * inv1)
        return np.clip(num / (self.Syy + EPS), 0.0, 1.0)


@torch.no_grad()
def accumulate_setting(setting: str, songs, acc: NLAccumulator) -> None:
    for song in songs:
        dry_path, wet_path, gr_path = pair_paths(song, setting)
        gr_obj = torch.load(gr_path, weights_only=False, map_location="cpu")
        gr_db_full = gr_obj["gr_db"].float()
        if gr_db_full.ndim == 1:
            gr_db_full = gr_db_full.unsqueeze(0)
        if gr_db_full.shape[0] > 1:
            gr_db_full = gr_db_full.mean(dim=0, keepdim=True)

        total = min(_pair_num_frames(dry_path, wet_path, SAMPLE_RATE), gr_db_full.shape[-1])
        for o in range(0, total, chunk_frames):
            stop = min(o + chunk_frames, total)
            dry, wet = _read_dry_wet_segment(dry_path, wet_path, o, stop, SAMPLE_RATE)
            L = min(dry.shape[-1], wet.shape[-1])
            if L < N_FFT:
                continue
            dry, wet = dry[..., :L], wet[..., :L]
            gr_db = gr_db_full[..., o : o + L].clamp(GR_DB_MIN, GR_DB_MAX)
            matched = dry * gr_db_to_gain(gr_db, clamp=False)
            acc.add(dry.squeeze(0).numpy(), wet.squeeze(0).numpy(), matched.squeeze(0).numpy())
        gc.collect()


# Run the single pass per setting.
results = {}
for si, setting in enumerate(SETTINGS, start=1):
    songs = sorted({song for (_, song, c) in USED_PAIRS if c == setting})
    acc = NLAccumulator(len(FREQS))
    print(f"[{si}/{len(SETTINGS)}] {setting}  songs={songs}")
    accumulate_setting(setting, songs, acc)
    results[setting] = {"acc": acc, "songs": songs}

print("\\nFrames per setting:")
for setting in SETTINGS:
    print(f"  {setting:48s} {results[setting]['acc'].n_frames:>8d}")

In [ ]:
F_MIN, F_MAX = 20.0, SAMPLE_RATE / 2
fmask = (FREQS >= F_MIN) & (FREQS <= F_MAX)
band = (FREQS >= 200) & (FREQS <= 8000)          # broadband summary band
cmap = plt.get_cmap("viridis", len(SETTINGS))


def short_label(setting: str) -> str:
    return (setting.replace("threshold_", "th").replace("_attack_", " a")
            .replace("_release_", " r").replace("_ratio_", " ratio"))


# Order settings by compression severity (low threshold + high ratio = hardest)
def severity(setting: str) -> tuple:
    d = dict(zip(setting.split("_")[::2], setting.split("_")[1::2]))
    return (float(d["threshold"]), -float(d["ratio"]))


SETTINGS_BY_SEV = sorted(SETTINGS, key=severity)
HARDEST = SETTINGS_BY_SEV[0]
print(f"Hardest setting (by threshold/ratio): {HARDEST}")

## §1 — Best-linear-model residual / NMSE

For every frequency bin the **optimal** per-bin LTI transfer is `H1 = Sxy/Sxx`;
the output power it leaves unexplained is `Syy·(1−γ²)`. Summed over the band this
is the **NMSE of the best linear time-invariant model** — the share of the wet
that *no* fixed filter of the dry can reproduce. For a truly linear box this
floors near the recording noise; a compressor pushes it up, and **harder settings
push it higher**.

`true test-tone THD is not available on program material` — this NMSE is its
data-driven stand-in. Curves below: `10·log10(1−γ²)` per frequency (higher = more
non-LTI energy).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for setting in SETTINGS_BY_SEV:
    acc = results[setting]["acc"]
    unexp_db = 10 * np.log10(acc.linear_unexplained() + EPS)
    ax.semilogx(FREQS[fmask], unexp_db[fmask], lw=1.2, label=short_label(setting))
ax.set_xlabel("frequency (Hz)")
ax.set_ylabel(r"non-LTI energy  $10\log_{10}(1-\gamma^2)$  (dB)")
ax.set_title("§1  Best-linear-model residual per frequency (higher = more nonlinear / time-varying)")
ax.set_xlim(F_MIN, F_MAX)
ax.grid(True, which="both", alpha=0.3)
ax.legend(fontsize=7, ncol=2, loc="upper right")
plt.tight_layout(); plt.show()

print("Broadband (200 Hz-8 kHz) best-LTI NMSE, ordered hardest -> softest:")
for setting in SETTINGS_BY_SEV:
    acc = results[setting]["acc"]
    nmse = np.sum(acc.Syy[band] * acc.linear_unexplained()[band]) / (np.sum(acc.Syy[band]) + EPS)
    print(f"  {short_label(setting):34s}  NMSE = {10*np.log10(nmse + EPS):7.2f} dB")

## §2 — Gain-matched residual spectrum

Remove the compressor's level/dynamics with the **known** broadband GR envelope
`g(t)` (the grey-box front-end used across this project), then inspect the
spectrum of the leftover `r = wet − g·dry`. A pure broadband gain would leave
`r ≈ 0`; what actually remains is the compressor's frequency-dependent
**coloration + harmonic distortion**. Curves: residual-to-output ratio
`10·log10(Srr/Syy)` per frequency.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for setting in SETTINGS_BY_SEV:
    acc = results[setting]["acc"]
    dr_db = 10 * np.log10(acc.distortion_ratio() + EPS)
    ax.semilogx(FREQS[fmask], dr_db[fmask], lw=1.2, label=short_label(setting))
ax.set_xlabel("frequency (Hz)")
ax.set_ylabel(r"residual / output  $10\log_{10}(S_{rr}/S_{yy})$  (dB)")
ax.set_title("Gain-matched residual spectrum  (coloration + harmonic distortion left after broadband GR)")
ax.set_xlim(F_MIN, F_MAX)
ax.grid(True, which="both", alpha=0.3)
ax.legend(fontsize=7, ncol=2, loc="upper right")
plt.tight_layout(); plt.show()

### Listen — gain-matched residual

The residual `r = wet − g·dry` is exactly the time-domain signal whose spectrum
the cell above plots (the STFT is linear, so `STFT(r) = Y − M`). Subtracting the
broadband gain `g(t)` removes the compressor's level/dynamics and leaves only its
**frequency-dependent coloration + harmonic distortion** — which you can now hear.

The players below render a short excerpt of the hardest setting. The first three (dry,
wet, gain-matched) share a common scale so the compression stays audible across
them; the **residual is peak-normalized** because it is much quieter than the
output (see the printed residual/output level in dB).

In [ ]:
# ---- Listen to the gain-matched residual  r = wet − g·dry  (the audible nonlinearity) ----
from IPython.display import Audio, display

LISTEN_SECONDS = 25.0      # excerpt length rendered as audio players
LISTEN_OFFSET_SEC = 30.0   # skip in to land on musical material (clamped to fit the file)


def _clamp_offset(offset_sec, total, seconds):
    o = int(round(offset_sec * SAMPLE_RATE))
    return min(o, max(total - int(round(seconds * SAMPLE_RATE)), 0))


def residual_excerpt(setting, seconds=LISTEN_SECONDS, offset_sec=LISTEN_OFFSET_SEC):
    """Time-domain dry / wet / gain-matched / residual (wet − g·dry) excerpt
    for the first song of one setting."""
    song = results[setting]["songs"][0]
    dry_path, wet_path, gr_path = pair_paths(song, setting)
    gr_obj = torch.load(gr_path, weights_only=False, map_location="cpu")
    gr_db_full = gr_obj["gr_db"].float()
    if gr_db_full.ndim == 1:
        gr_db_full = gr_db_full.unsqueeze(0)
    if gr_db_full.shape[0] > 1:
        gr_db_full = gr_db_full.mean(dim=0, keepdim=True)
    total = min(_pair_num_frames(dry_path, wet_path, SAMPLE_RATE), gr_db_full.shape[-1])
    o = _clamp_offset(offset_sec, total, seconds)
    stop = min(o + int(round(seconds * SAMPLE_RATE)), total)
    dry, wet = _read_dry_wet_segment(dry_path, wet_path, o, stop, SAMPLE_RATE)
    L = min(dry.shape[-1], wet.shape[-1])
    dry, wet = dry[..., :L], wet[..., :L]
    gr_db = gr_db_full[..., o:o + L].clamp(GR_DB_MIN, GR_DB_MAX)
    matched = dry * gr_db_to_gain(gr_db, clamp=False)
    res = wet - matched
    sq = lambda t: t.squeeze(0).numpy()
    return sq(dry), sq(wet), sq(matched), sq(res)


def play_residual(item, title=None):
    """Render dry / wet / gain-matched / residual audio players for one item."""
    dry, wet, matched, res = residual_excerpt(item)
    rms = lambda a: float(np.sqrt(np.mean(a ** 2)) + EPS)
    print(f"{title or item}")
    print(f"  residual/output = {20*np.log10(rms(res)/rms(wet)):5.1f} dB   "
          f"(peaks: wet {np.max(np.abs(wet)):.3f}, residual {np.max(np.abs(res)):.4f})")
    shared = max(np.max(np.abs(dry)), np.max(np.abs(wet)), np.max(np.abs(matched)), EPS)
    rpk = max(np.max(np.abs(res)), EPS)
    for name, sig, sc in [
        ("dry  x", dry, shared),
        ("wet  y", wet, shared),
        ("gain-matched  g·x   (broadband GR removed)", matched, shared),
        ("residual  r = y − g·x   [peak-normalized — the audible nonlinearity]", res, rpk),
    ]:
        print("  " + name)
        display(Audio(sig / sc, rate=SAMPLE_RATE, normalize=False))


# Listen to the hardest setting (set to any key in SETTINGS_BY_SEV).
play_residual(HARDEST, title=f"{short_label(HARDEST)}  (song {results[HARDEST]['songs'][0]})")


## §3 — Envelope multiple-coherence

Add the envelope-modulated input `m = g·dry` as a **second regressor** and compute
the multiple coherence `γ²_{y:(x,m)}`. Because `g(t)` varies *within* each window,
`STFT(g·x) ≠ g·STFT(x)` — `m` carries genuinely new (modulation-spread) structure,
so the 2×2 cross-spectral matrix is well-conditioned. The **gap**
`γ²_{y:(x,m)} − γ²_{yx}` is the output power explained *only* once the time-varying
gain is admitted — a direct, per-frequency signature of the envelope-driven
nonlinearity. Top: ordinary vs multiple coherence for the hardest setting.
Bottom: the gap for every setting.

In [ ]:
fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(11, 9), sharex=True)

acc_h = results[HARDEST]["acc"]
ax_top.semilogx(FREQS[fmask], acc_h.coherence()[fmask], lw=1.4,
                color="#1f77b4", label=r"ordinary  $\gamma^2_{yx}$")
ax_top.semilogx(FREQS[fmask], acc_h.multiple_coherence()[fmask], lw=1.4,
                color="#d62728", label=r"multiple  $\gamma^2_{y:(x,m)}$")
ax_top.set_ylabel("coherence")
ax_top.set_ylim(0, 1.02)
ax_top.set_title(f"§3  Ordinary vs multiple coherence — hardest setting ({short_label(HARDEST)})")
ax_top.grid(True, which="both", alpha=0.3)
ax_top.legend(loc="lower left")

for setting in SETTINGS_BY_SEV:
    acc = results[setting]["acc"]
    gap = acc.multiple_coherence() - acc.coherence()
    ax_bot.semilogx(FREQS[fmask], gap[fmask], lw=1.2, label=short_label(setting))
ax_bot.set_xlabel("frequency (Hz)")
ax_bot.set_ylabel(r"$\gamma^2_{y:(x,m)} - \gamma^2_{yx}$")
ax_bot.set_title("Coherence gap from the envelope-modulated input (higher = stronger time-varying-gain nonlinearity)")
ax_bot.set_xlim(F_MIN, F_MAX)
ax_bot.grid(True, which="both", alpha=0.3)
ax_bot.legend(fontsize=7, ncol=2, loc="upper right")
plt.tight_layout(); plt.show()

## §4 — Bicoherence (quadratic phase coupling)

The third-order analogue of coherence:

$$b^2(f_1,f_2)=\frac{\left|\;\mathbb{E}\!\left[X(f_1)X(f_2)X^*(f_1{+}f_2)\right]\right|^2}
{\mathbb{E}|X(f_1)X(f_2)|^2\;\cdot\;\mathbb{E}|X(f_1{+}f_2)|^2}.$$

It is ~0 for a linear process with independent-phase components and rises toward 1
where a **quadratic** nonlinearity phase-locks `f_1`, `f_2` and their sum.
Computed on one excerpt of the hardest setting, decimated for tractability. The
**wet** map showing structure absent from the **dry** map is nonlinearly-generated
coupling — proof the box is not linear, independent of any envelope model.

In [ ]:
# Decimated, single excerpt of the hardest setting (bicoherence is O(F^2)).
BIC_SR = 11025
BIC_NFFT = 256
BIC_HOP = BIC_NFFT // 2
BIC_SONG = results[HARDEST]["songs"][0]
BIC_DUR_SEC = 60.0

_dry_path, _wet_path, _ = pair_paths(BIC_SONG, HARDEST)
_total = _pair_num_frames(_dry_path, _wet_path, SAMPLE_RATE)
_stop = min(_total, int(BIC_DUR_SEC * SAMPLE_RATE))
_dry, _wet = _read_dry_wet_segment(_dry_path, _wet_path, 0, _stop, SAMPLE_RATE)
dry_ds = librosa.resample(_dry.squeeze(0).numpy(), orig_sr=SAMPLE_RATE, target_sr=BIC_SR)
wet_ds = librosa.resample(_wet.squeeze(0).numpy(), orig_sr=SAMPLE_RATE, target_sr=BIC_SR)


def bicoherence(sig, nfft, hop):
    S = librosa.stft(np.ascontiguousarray(sig, np.float32), n_fft=nfft,
                     hop_length=hop, window="hann", center=False)   # [F, T]
    F = S.shape[0]
    idx = np.arange(F)
    s = idx[:, None] + idx[None, :]
    valid = s < F
    s_clip = np.where(valid, s, 0)
    num = np.zeros((F, F), np.complex128)
    d1 = np.zeros((F, F), np.float64)
    d2 = np.zeros((F, F), np.float64)
    for t in range(S.shape[1]):
        Xt = S[:, t]
        prod = np.outer(Xt, Xt)            # X(f1) X(f2)
        Xsum = Xt[s_clip]                  # X(f1+f2)
        num += prod * np.conj(Xsum)
        d1 += np.abs(prod) ** 2
        d2 += np.abs(Xsum) ** 2
    b2 = (np.abs(num) ** 2) / (d1 * d2 + EPS)
    b2[~valid] = np.nan
    return b2, librosa.fft_frequencies(sr=BIC_SR, n_fft=nfft)


b2_dry, bfreq = bicoherence(dry_ds, BIC_NFFT, BIC_HOP)
b2_wet, _ = bicoherence(wet_ds, BIC_NFFT, BIC_HOP)
print(f"Bicoherence: {BIC_SONG} @ {HARDEST}  | {BIC_SR} Hz, n_fft={BIC_NFFT}, "
      f"{b2_dry.shape[0]} bins, excerpt {_stop/SAMPLE_RATE:.0f}s")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, data, title in zip(
    axes,
    [b2_dry, b2_wet, b2_wet - b2_dry],
    ["dry (input)  $b^2$", "wet (output)  $b^2$", "wet $-$ dry  (nonlinearly generated)"],
):
    pm = ax.pcolormesh(bfreq, bfreq, data, shading="auto", cmap="magma",
                       vmin=0, vmax=(np.nanmax(b2_wet) if "generated" not in title else None))
    ax.set_title(title)
    ax.set_xlabel("$f_1$ (Hz)")
    ax.set_ylabel("$f_2$ (Hz)")
    fig.colorbar(pm, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()

print(f"mean bicoherence  dry={np.nanmean(b2_dry):.4f}   wet={np.nanmean(b2_wet):.4f}   "
      f"(wet/dry = {np.nanmean(b2_wet)/ (np.nanmean(b2_dry)+EPS):.2f}x)")

## Summary

Per setting (hardest → softest), the four nonlinearity read-outs over the
200 Hz–8 kHz band. Ordinary coherence is shown for reference — note how little it
moves while the targeted measures separate the settings.

In [ ]:
rows = []
for setting in SETTINGS_BY_SEV:
    acc = results[setting]["acc"]
    nmse = np.sum(acc.Syy[band] * acc.linear_unexplained()[band]) / (np.sum(acc.Syy[band]) + EPS)
    rows.append({
        "Setting": setting,
        "Frames": acc.n_frames,
        "ord. coh (ref)": float(np.mean(acc.coherence()[band])),
        "§1 best-LTI NMSE dB": 10 * np.log10(nmse + EPS),
        "§2 resid/out dB": float(np.mean(10 * np.log10(acc.distortion_ratio()[band] + EPS))),
        "§3 multi-coh gap": float(np.mean((acc.multiple_coherence() - acc.coherence())[band])),
    })
summary_df = pd.DataFrame(rows)
display(summary_df.round(4))

## §2b — Decomposing the residual: delay artifact vs linear coloration vs nonlinearity

The §2 residual `r = wet - g*dry` lumps together three very different things, and
the rising, **setting-independent** high-frequency tail in the §2 plot is the
giveaway that most of it is *not* coloration:

1. **A constant (possibly fractional-sample) delay** between the exported wet and
   the dry. A pure delay leaves a residual whose power rises with frequency
   (`~ sin^2(pi f tau)`) and is identical for every setting -- an artifact of
   export/resampling/latency, not the compressor. A real, zero-phase broadband
   gain `g(t)` cannot correct it.
2. **A fixed linear coloration** (a frequency response the broadband gain misses)
   -- removable by an EQ, i.e. by the *best per-bin LTI filter* `H1 = Sxy/Sxx`.
3. **Genuinely non-LTI energy** (nonlinear + time-varying gain) -- the *only* part
   a learned colour model can ever reproduce. This is the target ceiling.

Two additions separate them. **§2b.1** estimates the per-pair sub-sample delay,
realigns wet, and re-accumulates the gain-matched residual. **§2b.2** overlays
three residual-to-output spectra so the gaps read off directly: (a)->(b) is the
delay artifact, (b)->(c) is fixed linear coloration, and the (c) floor `1 - gamma^2`
is the nonlinear/time-varying part. Crucially, `1 - gamma^2` is *already* immune to
any LTI delay or EQ (the per-bin complex filter absorbs both), so comparing it to
§2 needs no alignment -- the alignment pass just attributes how much of the gap is
delay versus frequency response.

In [ ]:
# == §2b.1 -- Sub-sample alignment of wet to the gain-matched signal =========
# Estimate a constant (fractional-sample) delay per pair, realign wet, and
# re-accumulate the gain-matched residual S_rr so it is free of timing artifacts.

def _rfft_delay(x, tau):
    """Shift x by tau samples via an FFT phase ramp (positive tau = delay)."""
    n = len(x)
    X = np.fft.rfft(x)
    f = np.fft.rfftfreq(n)
    return np.fft.irfft(X * np.exp(-2j * np.pi * f * tau), n).astype(np.float32)


def _gcc_phat_lag(sig, ref, max_lag=64, upsample=32):
    """Sub-sample lag (samples) maximising the PHAT cross-correlation of sig vs ref."""
    n = len(sig) + len(ref)
    nfft = 1 << int(np.ceil(np.log2(n)))
    R = np.fft.rfft(sig, nfft) * np.conj(np.fft.rfft(ref, nfft))
    R /= np.abs(R) + 1e-12
    cc = np.fft.irfft(R, nfft * upsample)            # sinc-interpolated xcorr
    m = int(max_lag * upsample)
    cc = np.concatenate((cc[-m:], cc[: m + 1]))
    return (np.argmax(cc) - m) / upsample


def _load_gr_mono(gr_path):
    gr = torch.load(gr_path, weights_only=False, map_location="cpu")["gr_db"].float()
    if gr.ndim == 1:
        gr = gr.unsqueeze(0)
    if gr.shape[0] > 1:
        gr = gr.mean(dim=0, keepdim=True)
    return gr


def estimate_pair_delay(song, setting, probe_sec=40.0, max_lag=64):
    """Signed delay d (samples) so that _rfft_delay(wet, d) best matches m = g*dry."""
    dry_path, wet_path, gr_path = pair_paths(song, setting)
    gr = _load_gr_mono(gr_path)
    total = min(_pair_num_frames(dry_path, wet_path, SAMPLE_RATE), gr.shape[-1])
    stop = min(total, int(probe_sec * SAMPLE_RATE))
    dry, wet = _read_dry_wet_segment(dry_path, wet_path, 0, stop, SAMPLE_RATE)
    L = min(dry.shape[-1], wet.shape[-1])
    dry, wet = dry[..., :L], wet[..., :L]
    m = (dry * gr_db_to_gain(gr[..., :L].clamp(GR_DB_MIN, GR_DB_MAX), clamp=False)).squeeze(0).numpy()
    w = wet.squeeze(0).numpy()
    tau = _gcc_phat_lag(w, m, max_lag=max_lag)
    # GCC sign convention is easy to flip; keep whichever shift of wet best matches
    # m, which makes the estimate convention-proof.
    cand = sorted({0.0, tau, -tau}, key=lambda d: float(np.mean((_rfft_delay(w, d) - m) ** 2)))
    return cand[0]


@torch.no_grad()
def accumulate_aligned(setting, songs):
    """Re-accumulate S_rr (and the matching S_yy) with wet realigned to the dry grid."""
    Srr = np.zeros(len(FREQS)); Syy = np.zeros(len(FREQS)); delays = []
    for song in songs:
        d = estimate_pair_delay(song, setting)
        delays.append(d)
        dry_path, wet_path, gr_path = pair_paths(song, setting)
        gr = _load_gr_mono(gr_path)
        total = min(_pair_num_frames(dry_path, wet_path, SAMPLE_RATE), gr.shape[-1])
        for o in range(0, total, chunk_frames):
            stop = min(o + chunk_frames, total)
            dry, wet = _read_dry_wet_segment(dry_path, wet_path, o, stop, SAMPLE_RATE)
            L = min(dry.shape[-1], wet.shape[-1])
            if L < N_FFT:
                continue
            dry, wet = dry[..., :L], wet[..., :L]
            grc = gr[..., o:o + L].clamp(GR_DB_MIN, GR_DB_MAX)
            m = (dry * gr_db_to_gain(grc, clamp=False)).squeeze(0).numpy()
            w = _rfft_delay(wet.squeeze(0).numpy(), d)
            M = _stft(m); Wl = _stft(w)
            Syy += np.sum(np.abs(Wl) ** 2, axis=1)
            Srr += np.sum(np.abs(Wl - M) ** 2, axis=1)
        gc.collect()
    return Srr, Syy, float(np.mean(delays))


print("Estimating sub-sample delay + re-accumulating aligned residual per setting...")
for si, setting in enumerate(SETTINGS_BY_SEV, start=1):
    songs = results[setting]["songs"]
    Srr_a, Syy_a, d_mean = accumulate_aligned(setting, songs)
    results[setting]["aligned"] = {"Srr": Srr_a, "Syy": Syy_a, "delay": d_mean}
    print(f"[{si:2d}/{len(SETTINGS_BY_SEV)}] {short_label(setting):34s} "
          f"delay = {d_mean:+7.3f} samples  ({d_mean / SAMPLE_RATE * 1e3:+.4f} ms)")

In [ ]:
# == §2b.2 -- Three-way decomposition of the residual ======================
#   (a) gain-matched, no alignment        10log10(S_rr / S_yy)        [the §2 curve]
#   (b) gain-matched, sub-sample aligned  10log10(S_rr_aligned / S_yy)
#   (c) best per-bin LTI floor            10log10(1 - gamma^2)        [the §1 floor]
# (a)->(b) gap = pure delay artifact removed by alignment.
# (b)->(c) gap = fixed linear coloration an EQ would remove.
# (c) floor    = nonlinear + time-varying energy = the learnable colour ceiling.
band_lf = (FREQS >= 20) & (FREQS < 500)

acc = results[HARDEST]["acc"]
al = results[HARDEST]["aligned"]
no_align = 10 * np.log10(acc.distortion_ratio() + EPS)
aligned = 10 * np.log10(al["Srr"] / (al["Syy"] + EPS) + EPS)
best_lti = 10 * np.log10(acc.linear_unexplained() + EPS)

fig, ax = plt.subplots(figsize=(11, 5))
ax.semilogx(FREQS[fmask], no_align[fmask], lw=1.3, color="#9467bd",
            label=r"(a) gain-matched, no alignment  $S_{rr}/S_{yy}$")
ax.semilogx(FREQS[fmask], aligned[fmask], lw=1.3, color="#1f77b4",
            label=r"(b) gain-matched, sub-sample aligned")
ax.semilogx(FREQS[fmask], best_lti[fmask], lw=1.6, color="#d62728",
            label=r"(c) best per-bin LTI floor  $1-\gamma^2$")
ax.set_xlabel("frequency (Hz)")
ax.set_ylabel(r"residual / output  (dB)")
ax.set_title(f"§2b  Residual decomposition -- hardest setting ({short_label(HARDEST)}, "
             f"delay {al['delay']:+.2f} samp)\n"
             "(a)->(b): delay artifact    (b)->(c): fixed linear coloration    "
             "(c): nonlinear + time-varying (learnable)")
ax.set_xlim(F_MIN, F_MAX)
ax.grid(True, which="both", alpha=0.3)
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()


def _band_db(num, den, mask):
    return 10 * np.log10(np.sum(num[mask]) / (np.sum(den[mask]) + EPS) + EPS)


rows = []
for setting in SETTINGS_BY_SEV:
    a = results[setting]["acc"]; al = results[setting]["aligned"]
    nlti = a.linear_unexplained() * a.Syy            # non-LTI residual power per bin
    for bname, bmask in [("LF<500", band_lf), ("200-8k", band)]:
        rows.append({
            "Setting": short_label(setting), "Band": bname,
            "delay samp": round(al["delay"], 3),
            "(a) no-align dB": _band_db(a.Srr, a.Syy, bmask),
            "(b) aligned dB": _band_db(al["Srr"], al["Syy"], bmask),
            "(c) best-LTI dB": _band_db(nlti, a.Syy, bmask),
        })
decomp_df = pd.DataFrame(rows)
print("Residual decomposition (dB residual/output). (c) is the learnable nonlinear ceiling.")
display(decomp_df.round(2))